# 🥈 Silver Layer Incremental Load — Bronze to Silver Transformation

This notebook transforms raw Bronze data into cleaned, standardized Silver tables.

---

## Purpose
This is the **main ETL notebook** that processes new Bronze data and loads it into Silver tables with:
- Data type conversions and standardization
- Code mapping using lookup tables
- Data quality validation and logging
- Incremental loading (processes only new data)

---

## Execution Frequency
- **Daily/Hourly**: Run when new Bronze data arrives
- **Incremental**: Only processes new Bronze records since last run
- **Orchestrated**: Designed to run as part of a pipeline

---

## Design Principles
- **Incremental by default** — watermark-based tracking prevents reprocessing
- **Snowpark-first** — DataFrames for transformations, SQL only when necessary
- **Clean and modular** — each table has dedicated transform + validate cells
- **Data quality focused** — validation after each load, issues logged to DQ_LOG
- **Simple and readable** — straightforward logic, clear variable names

---

## Transformations

This notebook transforms **13 Bronze tables** into **16 Silver tables**:

| Bronze Table | Silver Table(s) | Type |
|--------------|-----------------|------|
| WEB__WS_ORDERS | WEB_ORDERS | 1:1 (deduplicated) |
| WEB__WS_ORDER_ITEMS | WEB_ORDER_ITEMS | 1:1 |
| WEB__WS_CUSTOMERS | WEB_CUSTOMERS | 1:1 |
| MOB__MOBILE_ORDERS | MOB_ORDERS, MOB_ORDER_ITEMS, MOB_CUSTOMERS | 1:Many (flatten JSON) |
| WHL__WHOLESALE_ORDERS | WHL_ORDERS, WHL_ORDER_ITEMS, WHL_CUSTOMERS | 1:Many (flatten XML) |
| MKT__MARKETPLACE_ORDERS | MKT_ORDERS | 1:1 |
| PAY__TRANSACTIONS | PAY_TRANSACTIONS | 1:1 (deduplicated) |
| SHP__SHIPMENTS | SHP_SHIPMENTS | 1:1 |
| SUP__SUPPORT_TICKETS | SUP_TICKETS | 1:1 |
| MKTG__CAMPAIGNS | MKTG_CAMPAIGNS | 1:1 |
| REV__REVIEWS | REV_REVIEWS | 1:1 |
| APP__APP_EVENTS | APP_EVENTS | 1:1 |

---

## Prerequisites
- `SILVER_SETUP.ipynb` must be run first to create lookup tables
- Bronze data must be loaded into `BRONZE.*` schema

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Imports & Session Setup
# ══════════════════════════════════════════════════════════════════════════════

from datetime import datetime, timezone
from snowflake.snowpark import Session
from snowflake.snowpark.functions import (
    lit, col, current_timestamp, sql_expr,
    when, coalesce, upper, lower, trim, round as spark_round,
    concat, split, regexp_replace, lpad, length, substring,
    row_number, max as spark_max, count as spark_count, sum as spark_sum
)
from snowflake.snowpark import Window
from snowflake.snowpark.types import (
    StructType, StructField, StringType, IntegerType, 
    DecimalType, BooleanType, DateType, TimestampType
)

# Get active Snowflake session
session = get_active_session()

print("=" * 80)
print("SILVER LAYER INCREMENTAL LOAD — Bronze to Silver Transformation")
print("=" * 80)
print(f"Database  : {session.get_current_database()}")
print(f"Schema    : {session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")
print(f"Timestamp : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

---

## Configuration

Define schema names and execution settings.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Configuration
# ══════════════════════════════════════════════════════════════════════════════

BRONZE_SCHEMA = "BRONZE"
SILVER_SCHEMA = "SILVER"

# Execution tracking
RUN_START = datetime.now(timezone.utc)
EXECUTION_STATS = []

print(f"Bronze Schema: {BRONZE_SCHEMA}")
print(f"Silver Schema: {SILVER_SCHEMA}")
print(f"Run started:   {RUN_START.strftime('%Y-%m-%d %H:%M:%S UTC')}")

---

## Helper Functions

Reusable utilities for incremental loading, validation, and data quality logging.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Helper Functions
# ══════════════════════════════════════════════════════════════════════════════

def get_incremental_watermark(session, silver_table_name):
    """
    Get the most recent Bronze load timestamp from a Silver table.
    This is used to identify new Bronze records to process.
    
    Returns: Timestamp of last processed Bronze record, or '1900-01-01' if table is empty.
    """
    try:
        result = (session.table(f"{SILVER_SCHEMA}.{silver_table_name}")
                  .select(spark_max(col("_bronze_load_timestamp")).alias("max_ts"))
                  .collect())
        
        max_ts = result[0]["MAX_TS"]
        if max_ts is None:
            return "1900-01-01 00:00:00"
        return max_ts
    except Exception:
        # Table doesn't exist yet - return earliest possible date
        return "1900-01-01 00:00:00"


def log_dq_issue(session, source_system, source_table, target_table, issue_type, 
                 issue_severity, record_id=None, field_name=None, 
                 original_value=None, resolution=None, details=None):
    """
    Log a data quality issue to SILVER.DQ_LOG.
    
    Args:
        issue_type: duplicate, missing_value, invalid_format, orphan_record, 
                   data_type_error, mapping_failed
        issue_severity: info, warning, error
    """
    try:
        log_sql = f"""
        INSERT INTO {SILVER_SCHEMA}.DQ_LOG (
            run_timestamp, source_system, source_table, target_table,
            issue_type, issue_severity, record_identifier, field_name,
            original_value, resolution_action, details
        )
        VALUES (
            CURRENT_TIMESTAMP(),
            '{source_system}',
            '{source_table}',
            '{target_table}',
            '{issue_type}',
            '{issue_severity}',
            {f"'{record_id}'" if record_id else 'NULL'},
            {f"'{field_name}'" if field_name else 'NULL'},
            {f"'{str(original_value)[:500]}'" if original_value else 'NULL'},
            {f"'{resolution}'" if resolution else 'NULL'},
            {f"'{details}'" if details else 'NULL'}
        )
        """
        session.sql(log_sql).collect()
    except Exception as e:
        print(f"  Warning: Could not log DQ issue: {e}")


def validate_table_structure(session, table_name, expected_columns):
    """
    Validate that a Silver table has the expected column structure.
    
    Args:
        expected_columns: Dict of {column_name: expected_type}
    
    Returns: True if valid, False otherwise
    """
    try:
        table_df = session.table(f"{SILVER_SCHEMA}.{table_name}")
        actual_schema = {field.name: str(field.datatype) for field in table_df.schema.fields}
        
        missing_cols = set(expected_columns.keys()) - set(actual_schema.keys())
        
        if missing_cols:
            print(f"  ✗ Missing columns: {missing_cols}")
            return False
        
        print(f"  ✓ All {len(expected_columns)} expected columns present")
        return True
        
    except Exception as e:
        print(f"  ✗ Validation error: {e}")
        return False


def validate_row_counts(session, table_name, expected_min=0, expected_max=None):
    """
    Validate row counts are within expected range.
    """
    try:
        count = session.table(f"{SILVER_SCHEMA}.{table_name}").count()
        
        if count < expected_min:
            print(f"  ✗ Row count {count:,} below minimum {expected_min:,}")
            return False
        
        if expected_max and count > expected_max:
            print(f"  ✗ Row count {count:,} exceeds maximum {expected_max:,}")
            return False
        
        print(f"  ✓ Row count: {count:,}")
        return True
        
    except Exception as e:
        print(f"  ✗ Count error: {e}")
        return False


def show_sample_data(session, table_name, num_rows=3):
    """
    Display sample rows from a Silver table.
    """
    try:
        print(f"\n  Sample data from {table_name}:")
        session.table(f"{SILVER_SCHEMA}.{table_name}").limit(num_rows).show()
    except Exception as e:
        print(f"  Could not show sample: {e}")


print("✓ Helper functions defined:")
print("  - get_incremental_watermark()")
print("  - log_dq_issue()")
print("  - validate_table_structure()")
print("  - validate_row_counts()")
print("  - show_sample_data()")

---

# Web Store Transformations

Transform 3 CSV-based Web Store tables from Bronze to Silver.

## Web Store — Orders

**Source**: `BRONZE.WEB__WS_ORDERS` (CSV from PostgreSQL export)

**Transformations**:
- Deduplicate ~3% duplicate rows (keep latest by `LAST_UPD_DT`)
- Convert VARCHAR dates to DATE type
- Convert VARCHAR amounts to DECIMAL
- Map single-char status codes to canonical values via lookup
- Convert Y/N flags to BOOLEAN
- Handle empty strings as NULL

**Target**: `SILVER.WEB_ORDERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_ORDERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WEB__WS_ORDERS → SILVER.WEB_ORDERS")
print("─" * 80)

# Get incremental watermark
watermark = get_incremental_watermark(session, "WEB_ORDERS")
print(f"Watermark: {watermark}")

# Read new Bronze records
bronze_df = (session.table(f"{BRONZE_SCHEMA}.WEB__WS_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create Silver table if not exists
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WEB_ORDERS (
        order_id                INTEGER,
        customer_id             INTEGER,
        order_date              DATE,
        order_status            VARCHAR(20),
        order_amount            DECIMAL(10, 2),
        currency_code           VARCHAR(3),
        payment_method          VARCHAR(50),
        ship_country            VARCHAR(2),
        promo_code              VARCHAR(50),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform data
    # Step 1: Deduplicate using ROW_NUMBER
    window_spec = Window.partition_by(col("ORD_NO")).order_by(col("LAST_UPD_DT").desc())
    
    deduped_df = (bronze_df
                  .with_column("_row_num", row_number().over(window_spec))
                  .filter(col("_row_num") == lit(1)))
    
    # Count duplicates for logging
    duplicates_removed = new_count - deduped_df.count()
    if duplicates_removed > 0:
        print(f"Duplicates removed: {duplicates_removed:,}")
        log_dq_issue(session, 'web', 'BRONZE.WEB__WS_ORDERS', 'SILVER.WEB_ORDERS',
                    'duplicate', 'warning', details=f'{duplicates_removed} duplicate rows removed')
    
    # Step 2: Join with status lookup
    status_lookup = session.table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS")
    
    silver_df = (deduped_df
                 .join(status_lookup,
                       (col("ORD_STAT_CD") == status_lookup["source_status_code"]) &
                       (lit("web") == status_lookup["source_system"]),
                       "left")
                 .select(
                     col("ORD_NO").cast(IntegerType()).alias("order_id"),
                     col("CUST_NO").cast(IntegerType()).alias("customer_id"),
                     col("ORD_DT").cast(DateType()).alias("order_date"),
                     status_lookup["canonical_status"].alias("order_status"),
                     col("NET_AMT_LC").cast(DecimalType(10, 2)).alias("order_amount"),
                     col("CURR_CD").alias("currency_code"),
                     col("PYMT_MTHD").alias("payment_method"),
                     col("SHIP_CNTRY").alias("ship_country"),
                     when(col("PROMO_CD") == lit(""), None)
                       .otherwise(col("PROMO_CD")).alias("promo_code"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("web").alias("source_system")
                 ))
    
    # Write to Silver
    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.WEB_ORDERS")
    
    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WEB_ORDERS")
    
    # Track stats
    EXECUTION_STATS.append({
        'table': 'WEB_ORDERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': duplicates_removed
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_ORDERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WEB_ORDERS")
print("─" * 80)

# Validate structure
expected_cols = {
    'order_id': 'IntegerType',
    'customer_id': 'IntegerType',
    'order_date': 'DateType',
    'order_status': 'StringType',
    'order_amount': 'DecimalType'
}
validate_table_structure(session, "WEB_ORDERS", expected_cols)

# Validate row counts
validate_row_counts(session, "WEB_ORDERS", expected_min=0)

# Check for NULL order_ids (should be 0)
null_check = (session.table(f"{SILVER_SCHEMA}.WEB_ORDERS")
              .filter(col("order_id").is_null())
              .count())
if null_check == 0:
    print(f"  ✓ No NULL order_ids")
else:
    print(f"  ✗ Found {null_check} NULL order_ids")

# Show sample
show_sample_data(session, "WEB_ORDERS", 3)

print("─" * 80)

---

## Web Store — Order Items

**Source**: `BRONZE.WEB__WS_ORDER_ITEMS` (CSV)

**Transformations**:
- Convert VARCHAR amounts to DECIMAL
- Convert Y/N return flag to BOOLEAN
- Map category codes (already canonical in source)
- No deduplication needed (order_item_id is unique)

**Target**: `SILVER.WEB_ORDER_ITEMS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_ORDER_ITEMS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WEB__WS_ORDER_ITEMS → SILVER.WEB_ORDER_ITEMS")
print("─" * 80)

watermark = get_incremental_watermark(session, "WEB_ORDER_ITEMS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.WEB__WS_ORDER_ITEMS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WEB_ORDER_ITEMS (
        order_item_id           INTEGER,
        order_id                INTEGER,
        product_sku             VARCHAR(50),
        product_name            VARCHAR(200),
        category_code           VARCHAR(10),
        quantity                INTEGER,
        unit_price              DECIMAL(10, 2),
        line_total              DECIMAL(10, 2),
        discount_percent        DECIMAL(10, 4),
        is_returned             BOOLEAN,
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform
    silver_df = (bronze_df
                 .select(
                     col("ORD_ITM_NO").cast(IntegerType()).alias("order_item_id"),
                     col("ORD_NO").cast(IntegerType()).alias("order_id"),
                     col("PROD_CD").alias("product_sku"),
                     col("PROD_NM").alias("product_name"),
                     col("CAT_CD").alias("category_code"),
                     col("QTY_ORDD").cast(IntegerType()).alias("quantity"),
                     col("UNIT_PRC_LC").cast(DecimalType(10, 2)).alias("unit_price"),
                     col("LN_TTL_LC").cast(DecimalType(10, 2)).alias("line_total"),
                     col("DISC_PCT").cast(DecimalType(10, 4)).alias("discount_percent"),
                     when(col("RTRN_FLG") == lit("Y"), True)
                       .otherwise(False).alias("is_returned"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("web").alias("source_system")
                 ))
    
    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.WEB_ORDER_ITEMS")
    
    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WEB_ORDER_ITEMS")
    
    EXECUTION_STATS.append({
        'table': 'WEB_ORDER_ITEMS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_ORDER_ITEMS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WEB_ORDER_ITEMS")
print("─" * 80)

validate_row_counts(session, "WEB_ORDER_ITEMS", expected_min=0)

# Verify no negative quantities or prices
negative_check = (session.table(f"{SILVER_SCHEMA}.WEB_ORDER_ITEMS")
                  .filter((col("quantity") < 0) | (col("unit_price") < 0))
                  .count())
if negative_check == 0:
    print(f"  ✓ No negative quantities or prices")
else:
    print(f"  ✗ Found {negative_check} rows with negative values")

show_sample_data(session, "WEB_ORDER_ITEMS", 3)
print("─" * 80)

---

## Web Store — Customers

**Source**: `BRONZE.WEB__WS_CUSTOMERS` (CSV)

**Transformations**:
- Standardize email (lowercase, trim)
- Convert VARCHAR dates to DATE
- Map account status codes (A/S/D)
- Map loyalty tier codes via lookup
- Convert Y/N marketing opt-in to BOOLEAN

**Target**: `SILVER.WEB_CUSTOMERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_CUSTOMERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WEB__WS_CUSTOMERS → SILVER.WEB_CUSTOMERS")
print("─" * 80)

watermark = get_incremental_watermark(session, "WEB_CUSTOMERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.WEB__WS_CUSTOMERS")
             .filter(col("_LOAD_TIMESTAMP") > lit (watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WEB_CUSTOMERS (
        customer_id             INTEGER,
        email_address           VARCHAR(200),
        first_name              VARCHAR(100),
        last_name               VARCHAR(100),
        phone_number            VARCHAR(50),
        country_code            VARCHAR(2),
        city_name               VARCHAR(100),
        postal_code             VARCHAR(20),
        currency_code           VARCHAR(3),
        registration_date       DATE,
        account_status          VARCHAR(20),
        loyalty_tier            VARCHAR(20),
        marketing_opt_in        BOOLEAN,
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Join with loyalty tier lookup
    tier_lookup = session.table(f"{SILVER_SCHEMA}.LKP_LOYALTY_TIER")
    
    silver_df = (bronze_df
                 .join(tier_lookup,
                       (col("TIER_CD") == tier_lookup["source_tier_code"]) &
                       (lit("web") == tier_lookup["source_system"]),
                       "left")
                 .select(
                     col("CUST_NO").cast(IntegerType()).alias("customer_id"),
                     lower(trim(col("EML_ADDR"))).alias("email_address"),
                     col("FRST_NM").alias("first_name"),
                     col("LST_NM").alias("last_name"),
                     col("PHN_NO").alias("phone_number"),
                     col("CNTRY_CD").alias("country_code"),
                     col("CITY_NM").alias("city_name"),
                     col("PSTL_CD").alias("postal_code"),
                     col("CURR_CD").alias("currency_code"),
                     col("REGST_DT").cast(DateType()).alias("registration_date"),
                     when(col("ACCT_STAT_CD") == lit("A"), lit("active"))
                       .when(col("ACCT_STAT_CD") == lit("S"), lit("suspended"))
                       .when(col("ACCT_STAT_CD") == lit("D"), lit("deleted"))
                       .otherwise(None).alias("account_status"),
                     tier_lookup["canonical_tier"].alias("loyalty_tier"),
                     when(col("MRKT_OPT_IN") == lit("Y"), True)
                       .otherwise(False).alias("marketing_opt_in"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("web").alias("source_system")
                 ))
    
    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.WEB_CUSTOMERS")
    
    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WEB_CUSTOMERS")
    
    EXECUTION_STATS.append({
        'table': 'WEB_CUSTOMERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEB_CUSTOMERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WEB_CUSTOMERS")
print("─" * 80)

validate_row_counts(session, "WEB_CUSTOMERS", expected_min=0)

# Check email format (should all be lowercase)
upper_emails = (session.table(f"{SILVER_SCHEMA}.WEB_CUSTOMERS")
                .filter(col("email_address") != lower(col("email_address")))
                .count())
if upper_emails == 0:
    print(f"  ✓ All emails are lowercase")
else:
    print(f"  ✗ Found {upper_emails} emails not lowercase")

show_sample_data(session, "WEB_CUSTOMERS", 3)
print("─" * 80)

---

# Mobile App Transformations

Transform nested JSON structure from Mobile App into 3 Silver tables.

## Mobile App — Orders

**Source**: `BRONZE.MOB__MOBILE_ORDERS` (JSON with nested structure)

**JSON Structure**: Each row has RAW_DATA VARIANT column containing full order object

**Transformations**:
- Extract order header fields from JSON
- Convert Unix milliseconds to TIMESTAMP
- Map status codes via lookup
- Round float amounts to fix precision
- Standardize email
- Handle NULL userId for guest checkouts (~8%)

**Target**: `SILVER.MOB_ORDERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_ORDERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.MOB__MOBILE_ORDERS → SILVER.MOB_ORDERS")
print("─" * 80)

watermark = get_incremental_watermark(session, "MOB_ORDERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.MOB__MOBILE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.MOB_ORDERS (
        order_id                INTEGER,
        source_order_id         VARCHAR(50),
        customer_id             VARCHAR(50),
        email_address           VARCHAR(200),
        order_date              TIMESTAMP_NTZ,
        order_status            VARCHAR(20),
        order_amount            DECIMAL(10, 2),
        promo_code              VARCHAR(50),
        platform_type           VARCHAR(20),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Join with status lookup
    status_lookup = session.table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS")
    
    # Transform - extract from JSON and convert types
    # Note: Using SQL expressions for JSON path navigation
    silver_df = (bronze_df
                 .with_column("order_status_raw", col("RAW_DATA")["orderStatus"])
                 .join(status_lookup,
                       (col("order_status_raw") == status_lookup["source_status_code"]) &
                       (lit("mobile") == status_lookup["source_system"]),
                       "left")
                 .select(
                     col("RAW_DATA")["internalOrderRef"].cast(IntegerType()).alias("order_id"),
                     col("RAW_DATA")["orderId"].cast(StringType()).alias("source_order_id"),
                     col("RAW_DATA")["customerInfo"]["userId"].cast(StringType()).alias("customer_id"),
                     lower(trim(col("RAW_DATA")["customerInfo"]["emailAddress"].cast(StringType()))).alias("email_address"),
                     # Convert Unix milliseconds to timestamp (divide by 1000)
                     sql_expr("TO_TIMESTAMP(RAW_DATA:orderTimestampMs::INTEGER / 1000)").alias("order_date"),
                     status_lookup["canonical_status"].alias("order_status"),
                     # Round float to 2 decimals to avoid precision issues
                     spark_round(col("RAW_DATA")["paymentInfo"]["totalAmountCharged"].cast(DecimalType(10, 2)), 2).alias("order_amount"),
                     when(col("RAW_DATA")["paymentInfo"]["promoCodeApplied"] == lit(""), None)
                       .otherwise(col("RAW_DATA")["paymentInfo"]["promoCodeApplied"].cast(StringType())).alias("promo_code"),
                     col("RAW_DATA")["deviceContext"]["platformType"].cast(StringType()).alias("platform_type"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("mobile").alias("source_system")
                 ))
    
    # Check for guest checkouts (NULL customer_id)
    guest_count = silver_df.filter(col("customer_id").is_null()).count()
    if guest_count > 0:
        print(f"Guest checkouts (NULL customer_id): {guest_count:,}")
        log_dq_issue(session, 'mobile', 'BRONZE.MOB__MOBILE_ORDERS', 'SILVER.MOB_ORDERS',
                    'missing_value', 'info', details=f'{guest_count} guest checkout orders')
    
    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.MOB_ORDERS")
    
    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.MOB_ORDERS")
    
    EXECUTION_STATS.append({
        'table': 'MOB_ORDERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_ORDERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.MOB_ORDERS")
print("─" * 80)

validate_row_counts(session, "MOB_ORDERS", expected_min=0)

# Check order_date is reasonable (not in future)
future_dates = (session.table(f"{SILVER_SCHEMA}.MOB_ORDERS")
                .filter(col("order_date") > current_timestamp())
                .count())
if future_dates == 0:
    print(f"  ✓ No future order dates")
else:
    print(f"  ✗ Found {future_dates} orders with future dates")

show_sample_data(session, "MOB_ORDERS", 3)
print("─" * 80)

---

## Mobile App — Order Items

**Source**: `BRONZE.MOB__MOBILE_ORDERS` (same JSON, flatten orderItems array)

**Transformation Strategy**: Use LATERAL FLATTEN to explode orderItems[] array

**Transformations**:
- Flatten nested orderItems array (1 Bronze row → N Silver rows)
- Extract item-level fields
- Map category labels via BRIDGE_CATEGORY
- Round float amounts

**Target**: `SILVER.MOB_ORDER_ITEMS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_ORDER_ITEMS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.MOB__MOBILE_ORDERS → SILVER.MOB_ORDER_ITEMS (flatten)")
print("─" * 80)

watermark = get_incremental_watermark(session, "MOB_ORDER_ITEMS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.MOB__MOBILE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.MOB_ORDER_ITEMS (
        order_id                INTEGER,
        product_sku             VARCHAR(50),
        product_name            VARCHAR(200),
        category_code           VARCHAR(10),
        quantity                INTEGER,
        unit_price              DECIMAL(10, 2),
        line_total              DECIMAL(10, 2),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Use SQL for FLATTEN since Snowpark DataFrame doesn't have direct flatten API
    # Use subquery to avoid FLATTEN on left side of LEFT JOIN
    flatten_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.MOB_ORDER_ITEMS
    SELECT
        t.order_id,
        t.product_sku,
        t.product_name,
        cat.category_code AS category_code,
        t.quantity,
        t.unit_price,
        t.line_total,
        t._bronze_source_file,
        t._bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'mobile' AS source_system
    FROM (
        SELECT
            bronze.RAW_DATA:internalOrderRef::INTEGER AS order_id,
            item.value:productIdentifier::VARCHAR AS product_sku,
            item.value:productName::VARCHAR AS product_name,
            item.value:categoryLabel::VARCHAR AS category_label,
            item.value:quantityOrdered::INTEGER AS quantity,
            ROUND(item.value:unitPriceLocal::DECIMAL(10,2), 2) AS unit_price,
            ROUND(item.value:lineTotalLocal::DECIMAL(10,2), 2) AS line_total,
            bronze._SOURCE_FILE AS _bronze_source_file,
            bronze._LOAD_TIMESTAMP AS _bronze_load_timestamp
        FROM {BRONZE_SCHEMA}.MOB__MOBILE_ORDERS bronze
        , LATERAL FLATTEN(input => bronze.RAW_DATA:orderItems) item
        WHERE bronze._LOAD_TIMESTAMP > '{watermark}'
    ) t
    LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CATEGORY cat
        ON t.category_label = cat.source_category
        AND cat.source_system = 'mobile'
    """
    
    session.sql(flatten_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.MOB_ORDER_ITEMS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.MOB_ORDER_ITEMS")
    print(f"  (Flattened from {new_count:,} Bronze orders)")
    
    EXECUTION_STATS.append({
        'table': 'MOB_ORDER_ITEMS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_ORDER_ITEMS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.MOB_ORDER_ITEMS")
print("─" * 80)

validate_row_counts(session, "MOB_ORDER_ITEMS", expected_min=0)

# Verify all have category codes
null_categories = (session.table(f"{SILVER_SCHEMA}.MOB_ORDER_ITEMS")
                   .filter(col("category_code").is_null())
                   .count())
if null_categories == 0:
    print(f"  ✓ All items have category codes")
else:
    print(f"  ⚠ {null_categories} items missing category codes")

show_sample_data(session, "MOB_ORDER_ITEMS", 3)
print("─" * 80)

In [ ]:
select * from mob_order_items limit 10

---

## Mobile App — Customers

**Source**: `BRONZE.MOB__MOBILE_ORDERS` (extract customerInfo, deduplicate)

**Transformations**:
- Extract customer info from JSON
- Deduplicate by userId (keep most recent)
- Split full name into first/last
- Map loyalty tier via lookup
- Standardize email

**Target**: `SILVER.MOB_CUSTOMERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_CUSTOMERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.MOB__MOBILE_ORDERS → SILVER.MOB_CUSTOMERS (extract)")
print("─" * 80)

watermark = get_incremental_watermark(session, "MOB_CUSTOMERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.MOB__MOBILE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.MOB_CUSTOMERS (
        customer_id             VARCHAR(50),
        email_address           VARCHAR(200),
        full_name               VARCHAR(200),
        first_name              VARCHAR(100),
        last_name               VARCHAR(100),
        loyalty_tier            VARCHAR(20),
        country_code            VARCHAR(2),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Join with loyalty tier lookup
    tier_lookup = session.table(f"{SILVER_SCHEMA}.LKP_LOYALTY_TIER")
    
    # Extract and deduplicate (keep most recent per userId)
    window_spec = Window.partition_by(col("RAW_DATA")["customerInfo"]["userId"]).order_by(col("_LOAD_TIMESTAMP").desc())
    
    silver_df = (bronze_df
                 .with_column("loyalty_tier_raw", col("RAW_DATA")["customerInfo"]["loyaltyTier"])
                 .with_column("_row_num", row_number().over(window_spec))
                 .filter(col("_row_num") == lit(1))
                 # Filter out guest checkouts
                 .filter(col("RAW_DATA")["customerInfo"]["userId"].is_not_null())
                 .join(tier_lookup,
                       (col("loyalty_tier_raw") == tier_lookup["source_tier_code"]) &
                       (lit("mobile") == tier_lookup["source_system"]),
                       "left")
                 .with_column("full_name_val", col("RAW_DATA")["customerInfo"]["fullName"].cast(StringType()))
                 .select(
                     col("RAW_DATA")["customerInfo"]["userId"].cast(StringType()).alias("customer_id"),
                     lower(trim(col("RAW_DATA")["customerInfo"]["emailAddress"].cast(StringType()))).alias("email_address"),
                     col("full_name_val").alias("full_name"),
                     # Split full name - take first part as first name
                     split(col("full_name_val"), lit(" ")).getItem(0).alias("first_name"),
                     # Take last part as last name
                     split(col("full_name_val"), lit(" ")).getItem(1).alias("last_name"),
                     tier_lookup["canonical_tier"].alias("loyalty_tier"),
                     col("RAW_DATA")["customerInfo"]["countryCode"].cast(StringType()).alias("country_code"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("mobile").alias("source_system")
                 ))
    
    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.MOB_CUSTOMERS")
    
    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.MOB_CUSTOMERS")
    
    EXECUTION_STATS.append({
        'table': 'MOB_CUSTOMERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': new_count - rows_inserted
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MOB_CUSTOMERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.MOB_CUSTOMERS")
print("─" * 80)

validate_row_counts(session, "MOB_CUSTOMERS", expected_min=0)

# Check all have email
null_emails = (session.table(f"{SILVER_SCHEMA}.MOB_CUSTOMERS")
               .filter(col("email_address").is_null())
               .count())
if null_emails == 0:
    print(f"  ✓ All customers have email addresses")
else:
    print(f"  ✗ Found {null_emails} customers without email")

show_sample_data(session, "MOB_CUSTOMERS", 3)
print("─" * 80)

---

# Wholesale Transformations

Transform nested XML structure from SAP ERP into 3 Silver tables.

## Wholesale — Orders

**Source**: `BRONZE.WHL__WHOLESALE_ORDERS` (XML with nested LineItem elements)

**XML Structure**: Each row has RAW_DATA VARIANT column containing XML document

**Transformations**:
- Extract order header from XML
- Convert DD/MM/YYYY dates to DATE
- Use GrossOrderValueInclTax (not NetOrderValueExclTax)
- Handle empty string promo codes
- Map status codes via lookup
- Extract payment terms (not payment method)

**Target**: `SILVER.WHL_ORDERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_ORDERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WHL__WHOLESALE_ORDERS → SILVER.WHL_ORDERS")
print("─" * 80)

watermark = get_incremental_watermark(session, "WHL_ORDERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WHL_ORDERS (
        order_id                INTEGER,
        buyer_po_number         VARCHAR(50),
        customer_id             VARCHAR(50),
        email_address           VARCHAR(200),
        order_date              DATE,
        order_status            VARCHAR(20),
        order_amount            DECIMAL(10, 2),
        order_amount_excl_tax   DECIMAL(10, 2),
        tax_amount              DECIMAL(10, 2),
        payment_terms           VARCHAR(20),
        promo_code              VARCHAR(50),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # SPLIT_TO_TABLE splits on </PurchaseOrderRecord> - each chunk still has all inner tags intact
    # Use REGEXP to extract values from each chunk - no XML parsing needed
    insert_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.WHL_ORDERS
    SELECT
        t.order_id,
        t.buyer_po_number,
        t.customer_id,
        t.email_address,
        t.order_date,
        COALESCE(lkp.canonical_status, 'UNKNOWN') AS order_status,
        t.order_amount,
        t.order_amount_excl_tax,
        t.tax_amount,
        t.payment_terms,
        t.promo_code,
        t._bronze_source_file,
        t._bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'wholesale' AS source_system
    FROM (
        SELECT
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<InternalOrderNumber>[^<]*</InternalOrderNumber>'), '<[^>]*>', '')::INTEGER AS order_id,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<BuyerPurchaseOrderNumber>[^<]*</BuyerPurchaseOrderNumber>'), '<[^>]*>', '')::VARCHAR AS buyer_po_number,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WholesaleAccountCode>[^<]*</WholesaleAccountCode>'), '<[^>]*>', '')::VARCHAR AS customer_id,
            LOWER(TRIM(REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<BuyerContactEmail>[^<]*</BuyerContactEmail>'), '<[^>]*>', '')::VARCHAR)) AS email_address,
            TO_DATE(REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<OrderCreatedDate>[^<]*</OrderCreatedDate>'), '<[^>]*>', '')::VARCHAR, 'DD/MM/YYYY') AS order_date,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<OrderStatusCode>[^<]*</OrderStatusCode>'), '<[^>]*>', '')::VARCHAR AS order_status_code,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<GrossOrderValueInclTax>[^<]*</GrossOrderValueInclTax>'), '<[^>]*>', '')::DECIMAL(10,2) AS order_amount,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<NetOrderValueExclTax>[^<]*</NetOrderValueExclTax>'), '<[^>]*>', '')::DECIMAL(10,2) AS order_amount_excl_tax,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<TaxAmount>[^<]*</TaxAmount>'), '<[^>]*>', '')::DECIMAL(10,2) AS tax_amount,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<PaymentTermsCode>[^<]*</PaymentTermsCode>'), '<[^>]*>', '')::VARCHAR AS payment_terms,
            NULLIF(REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<PromotionalReferenceCode>[^<]*</PromotionalReferenceCode>'), '<[^>]*>', '')::VARCHAR, '') AS promo_code,
            bronze._SOURCE_FILE AS _bronze_source_file,
            bronze._LOAD_TIMESTAMP AS _bronze_load_timestamp
        FROM {BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS bronze,
        TABLE(SPLIT_TO_TABLE(bronze.RAW_DATA::VARCHAR, '</PurchaseOrderRecord>')) chunk
        WHERE bronze._LOAD_TIMESTAMP > '{watermark}'
        AND chunk.VALUE LIKE '%<InternalOrderNumber>%'
    ) t
    LEFT JOIN {SILVER_SCHEMA}.LKP_ORDER_STATUS lkp
        ON t.order_status_code = lkp.source_status_code
        AND lkp.source_system = 'wholesale'
    """
    
    session.sql(insert_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.WHL_ORDERS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WHL_ORDERS")
    print(f"  (Flattened from {new_count:,} Bronze XML file(s) containing multiple orders)")
    
    EXECUTION_STATS.append({
        'table': 'WHL_ORDERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_ORDERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WHL_ORDERS")
print("─" * 80)

validate_row_counts(session, "WHL_ORDERS", expected_min=0)

# Verify tax_amount + order_amount_excl_tax ≈ order_amount
# (Allow small rounding differences)
tax_mismatch = (session.table(f"{SILVER_SCHEMA}.WHL_ORDERS")
                .filter(sql_expr("ABS(tax_amount + order_amount_excl_tax - order_amount) > 0.02"))
                .count())
if tax_mismatch == 0:
    print(f"  ✓ Tax calculations are consistent")
else:
    print(f"  ⚠ {tax_mismatch} orders have tax calculation discrepancies")

show_sample_data(session, "WHL_ORDERS", 3)
print("─" * 80)

---

## Wholesale — Order Items

**Source**: `BRONZE.WHL__WHOLESALE_ORDERS` (flatten LineItem XML elements)

**Transformation Strategy**: Use LATERAL FLATTEN on XML LineItem array

**Transformations**:
- Flatten nested LineItem elements
- Map category codes via BRIDGE_CATEGORY
- Extract item-level fields

**Target**: `SILVER.WHL_ORDER_ITEMS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_ORDER_ITEMS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WHL__WHOLESALE_ORDERS → SILVER.WHL_ORDER_ITEMS (flatten)")
print("─" * 80)

watermark = get_incremental_watermark(session, "WHL_ORDER_ITEMS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WHL_ORDER_ITEMS (
        order_id                INTEGER,
        product_sku             VARCHAR(50),
        product_name            VARCHAR(200),
        category_code           VARCHAR(10),
        quantity                INTEGER,
        unit_price              DECIMAL(10, 2),
        line_total              DECIMAL(10, 2),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Two-level split: first by PurchaseOrderRecord, then by LineItem
    flatten_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.WHL_ORDER_ITEMS
    SELECT
        t.order_id,
        t.product_sku,
        t.product_name,
        cat.category_code AS category_code,
        t.quantity,
        t.unit_price,
        t.line_total,
        t._bronze_source_file,
        t._bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'wholesale' AS source_system
    FROM (
        SELECT
            oc.order_id,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<StockKeepingUnit>[^<]*</StockKeepingUnit>'), '<[^>]*>', '')::VARCHAR AS product_sku,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<ProductDescription>[^<]*</ProductDescription>'), '<[^>]*>', '')::VARCHAR AS product_name,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<ProductCategoryCode>[^<]*</ProductCategoryCode>'), '<[^>]*>', '')::VARCHAR AS category_code_source,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<QuantityOrdered>[^<]*</QuantityOrdered>'), '<[^>]*>', '')::INTEGER AS quantity,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<UnitPriceNetExclTax>[^<]*</UnitPriceNetExclTax>'), '<[^>]*>', '')::DECIMAL(10,2) AS unit_price,
            REGEXP_REPLACE(REGEXP_SUBSTR(item.VALUE, '<LineTotalNetExclTax>[^<]*</LineTotalNetExclTax>'), '<[^>]*>', '')::DECIMAL(10,2) AS line_total,
            oc._bronze_source_file,
            oc._bronze_load_timestamp
        FROM (
            SELECT
                REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<InternalOrderNumber>[^<]*</InternalOrderNumber>'), '<[^>]*>', '')::INTEGER AS order_id,
                chunk.VALUE AS order_xml,
                bronze._SOURCE_FILE AS _bronze_source_file,
                bronze._LOAD_TIMESTAMP AS _bronze_load_timestamp
            FROM {BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS bronze,
            TABLE(SPLIT_TO_TABLE(bronze.RAW_DATA::VARCHAR, '</PurchaseOrderRecord>')) chunk
            WHERE bronze._LOAD_TIMESTAMP > '{watermark}'
            AND chunk.VALUE LIKE '%<InternalOrderNumber>%'
        ) oc,
        TABLE(SPLIT_TO_TABLE(oc.order_xml, '</LineItem>')) item
        WHERE item.VALUE LIKE '%<StockKeepingUnit>%'
    ) t
    LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CATEGORY cat
        ON t.category_code_source = cat.source_category
        AND cat.source_system = 'wholesale'
    """
    
    session.sql(flatten_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.WHL_ORDER_ITEMS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WHL_ORDER_ITEMS")
    print(f"  (Flattened from {new_count:,} Bronze XML file(s))")
    
    EXECUTION_STATS.append({
        'table': 'WHL_ORDER_ITEMS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_ORDER_ITEMS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WHL_ORDER_ITEMS")
print("─" * 80)

validate_row_counts(session, "WHL_ORDER_ITEMS", expected_min=0)

show_sample_data(session, "WHL_ORDER_ITEMS", 3)
print("─" * 80)

---

## Wholesale — Customers

**Source**: `BRONZE.WHL__WHOLESALE_ORDERS` (extract buyer info, deduplicate)

**Transformations**:
- Extract buyer information from XML
- Deduplicate by WholesaleAccountCode
- Standardize email

**Target**: `SILVER.WHL_CUSTOMERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_CUSTOMERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.WHL__WHOLESALE_ORDERS → SILVER.WHL_CUSTOMERS (extract)")
print("─" * 80)

watermark = get_incremental_watermark(session, "WHL_CUSTOMERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.WHL_CUSTOMERS (
        customer_id             VARCHAR(50),
        email_address           VARCHAR(200),
        company_name            VARCHAR(200),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Split XML into individual PurchaseOrderRecords, extract customer info, deduplicate
    insert_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.WHL_CUSTOMERS
    SELECT
        customer_id,
        email_address,
        company_name,
        _bronze_source_file,
        _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'wholesale' AS source_system
    FROM (
        SELECT
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WholesaleAccountCode>[^<]*</WholesaleAccountCode>'), '<[^>]*>', '')::VARCHAR AS customer_id,
            LOWER(TRIM(REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<BuyerContactEmail>[^<]*</BuyerContactEmail>'), '<[^>]*>', '')::VARCHAR)) AS email_address,
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<BuyerCompanyName>[^<]*</BuyerCompanyName>'), '<[^>]*>', '')::VARCHAR AS company_name,
            bronze._SOURCE_FILE AS _bronze_source_file,
            bronze._LOAD_TIMESTAMP AS _bronze_load_timestamp,
            ROW_NUMBER() OVER (PARTITION BY 
                REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WholesaleAccountCode>[^<]*</WholesaleAccountCode>'), '<[^>]*>', '')
                ORDER BY bronze._LOAD_TIMESTAMP DESC
            ) AS rn
        FROM {BRONZE_SCHEMA}.WHL__WHOLESALE_ORDERS bronze,
        TABLE(SPLIT_TO_TABLE(bronze.RAW_DATA::VARCHAR, '</PurchaseOrderRecord>')) chunk
        WHERE bronze._LOAD_TIMESTAMP > '{watermark}'
        AND chunk.VALUE LIKE '%<WholesaleAccountCode>%'
    )
    WHERE rn = 1
    """
    
    session.sql(insert_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.WHL_CUSTOMERS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.WHL_CUSTOMERS")
    
    EXECUTION_STATS.append({
        'table': 'WHL_CUSTOMERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WHL_CUSTOMERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.WHL_CUSTOMERS")
print("─" * 80)

validate_row_counts(session, "WHL_CUSTOMERS", expected_min=0)

show_sample_data(session, "WHL_CUSTOMERS", 3)
print("─" * 80)

---

# Other Source Transformations

## Marketplace Orders

**Source**: `BRONZE.MKT__MARKETPLACE_ORDERS` (CSV, already flat)

**Note**: This table is **one row per line item** (not per order). Multiple rows belong to same amazon-order-id.

**Transformations**:
- Convert MM/DD/YYYY HH:MM:SS dates to TIMESTAMP
- Convert VARCHAR amounts to DECIMAL
- Map status codes via lookup
- Map category via BRIDGE_CATEGORY
- Handle empty string promo codes
- Note: Buyer PII is masked (email shows domain only)

**Target**: `SILVER.MKT_ORDERS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MKT_ORDERS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.MKT__MARKETPLACE_ORDERS → SILVER.MKT_ORDERS")
print("─" * 80)

watermark = get_incremental_watermark(session, "MKT_ORDERS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.MKT__MARKETPLACE_ORDERS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.MKT_ORDERS (
        order_id                INTEGER,
        amazon_order_id         VARCHAR(50),
        purchase_date           TIMESTAMP_NTZ,
        order_status            VARCHAR(20),
        fulfillment_channel     VARCHAR(10),
        buyer_email_masked      VARCHAR(100),
        buyer_name_masked       VARCHAR(100),
        product_sku             VARCHAR(50),
        asin                    VARCHAR(20),
        product_name            VARCHAR(200),
        category_code           VARCHAR(10),
        quantity                INTEGER,
        currency_code           VARCHAR(3),
        unit_price              DECIMAL(10, 2),
        line_total              DECIMAL(10, 2),
        promo_code              VARCHAR(50),
        marketplace_fee         DECIMAL(10, 2),
        ship_country            VARCHAR(2),
        payment_method          VARCHAR(50),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()

    status_lookup = session.table(f"{SILVER_SCHEMA}.LKP_ORDER_STATUS")
    category_bridge = session.table(f"{SILVER_SCHEMA}.BRIDGE_CATEGORY")

    silver_df = (bronze_df
                 .join(status_lookup,
                       (col('"order-status"') == status_lookup["source_status_code"]) &
                       (lit("marketplace") == status_lookup["source_system"]),
                       "left")
                 .join(category_bridge,
                       (col('"product-category"') == category_bridge["source_category"]) &
                       (lit("marketplace") == category_bridge["source_system"]),
                       "left")
                 .select(
                     col('"merchant-order-id"').cast(IntegerType()).alias("order_id"),
                     col('"amazon-order-id"').alias("amazon_order_id"),
                     col('"purchase-date"').cast(TimestampType()).alias("purchase_date"),
                     status_lookup["canonical_status"].alias("order_status"),
                     col('"fulfillment-channel"').alias("fulfillment_channel"),
                     col('"buyer-email"').alias("buyer_email_masked"),
                     col('"buyer-name"').alias("buyer_name_masked"),
                     col('"sku"').alias("product_sku"),
                     col('"asin"').alias("asin"),
                     col('"product-name"').alias("product_name"),
                     category_bridge["category_code"].alias("category_code"),
                     col('"quantity-purchased"').cast(IntegerType()).alias("quantity"),
                     col('"currency"').alias("currency_code"),
                     col('"item-price"').cast(DecimalType(10, 2)).alias("unit_price"),
                     col('"item-total"').cast(DecimalType(10, 2)).alias("line_total"),
                     when(col('"promotion-ids"') == lit(""), None)
                       .otherwise(col('"promotion-ids"')).alias("promo_code"),
                     col('"marketplace-fee"').cast(DecimalType(10, 2)).alias("marketplace_fee"),
                     col('"ship-country"').alias("ship_country"),
                     col('"payment-method-details"').alias("payment_method"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("marketplace").alias("source_system")
                 ))

    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.MKT_ORDERS")

    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.MKT_ORDERS")
    print(f"  Note: This table is one row per line item (not per order)")

    EXECUTION_STATS.append({
        'table': 'MKT_ORDERS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MKT_ORDERS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.MKT_ORDERS")
print("─" * 80)

validate_row_counts(session, "MKT_ORDERS", expected_min=0)

# Log that buyer email is masked
masked_count = (session.table(f"{SILVER_SCHEMA}.MKT_ORDERS")
                .filter(col("buyer_email_masked").like("%***%"))
                .count())
print(f"  ℹ {masked_count:,} rows have masked buyer email (expected per Amazon policy)")

show_sample_data(session, "MKT_ORDERS", 3)
print("─" * 80)

---

## Payment Transactions

**Source**: `BRONZE.PAY__TRANSACTIONS` (CSV)

**Note**: Covers WEB and MOBILE orders only (not Wholesale or Marketplace)

**Transformations**:
- Deduplicate payment retries (keep latest successful, or latest attempt)
- Convert VARCHAR amounts to DECIMAL
- Map status codes via LKP_PAYMENT_STATUS
- Map payment method codes via LKP_PAYMENT_METHOD
- Convert ISO 8601 dates to TIMESTAMP
- Convert 0/1 flags to BOOLEAN

**Special Handling**: ~8% of web/mobile orders may have no payment record (abandoned carts)

**Target**: `SILVER.PAY_TRANSACTIONS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PAY_TRANSACTIONS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.PAY__TRANSACTIONS → SILVER.PAY_TRANSACTIONS")
print("─" * 80)

watermark = get_incremental_watermark(session, "PAY_TRANSACTIONS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.PAY__TRANSACTIONS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.PAY_TRANSACTIONS (
        transaction_id          VARCHAR(50),
        order_id                INTEGER,
        source_channel          VARCHAR(20),
        payment_method_code     VARCHAR(10),
        payment_method          VARCHAR(50),
        gross_amount            DECIMAL(10, 2),
        fee_amount              DECIMAL(10, 2),
        net_amount              DECIMAL(10, 2),
        currency_code           VARCHAR(3),
        payment_status_code     INTEGER,
        payment_status          VARCHAR(20),
        is_successful           BOOLEAN,
        failure_reason_code     VARCHAR(10),
        processed_datetime      TIMESTAMP_TZ,
        is_first_transaction    BOOLEAN,
        gateway_region          VARCHAR(10),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()

    # Deduplicate: Keep latest successful, or latest attempt if all failed
    window_spec = (Window.partition_by(col('"external_ref"'))
                   .order_by(
                       when(col('"stat_cd"') == lit(1), 0).otherwise(1).asc(),
                       col('"proc_dt_utc"').desc()
                   ))

    deduped_df = (bronze_df
                  .with_column("_row_num", row_number().over(window_spec))
                  .filter(col("_row_num") == lit(1)))

    duplicates_removed = new_count - deduped_df.count()
    if duplicates_removed > 0:
        print(f"Payment retries removed: {duplicates_removed:,}")

    # Join with lookups
    status_lookup = session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_STATUS")
    method_lookup = session.table(f"{SILVER_SCHEMA}.LKP_PAYMENT_METHOD")

    silver_df = (deduped_df
                 .join(status_lookup,
                       col('"stat_cd"') == status_lookup["status_code"],
                       "left")
                 .join(method_lookup,
                       col('"pymt_mthd_cd"') == method_lookup["payment_method_code"],
                       "left")
                 .select(
                     col('"txn_ref_id"').alias("transaction_id"),
                     col('"external_ref"').cast(IntegerType()).alias("order_id"),
                     col('"source_channel"').alias("source_channel"),
                     col('"pymt_mthd_cd"').alias("payment_method_code"),
                     method_lookup["payment_method_name"].alias("payment_method"),
                     col('"gross_amt"').cast(DecimalType(10, 2)).alias("gross_amount"),
                     col('"fee_amt"').cast(DecimalType(10, 2)).alias("fee_amount"),
                     col('"net_amt_after_fees"').cast(DecimalType(10, 2)).alias("net_amount"),
                     col('"curr_cd"').alias("currency_code"),
                     col('"stat_cd"').cast(IntegerType()).alias("payment_status_code"),
                     status_lookup["status_name"].alias("payment_status"),
                     status_lookup["is_successful"].alias("is_successful"),
                     col('"failure_rsn_cd"').alias("failure_reason_code"),
                     col('"proc_dt_utc"').cast(TimestampType()).alias("processed_datetime"),
                     when(col('"is_frst_txn_flg"') == lit("1"), True)
                       .otherwise(False).alias("is_first_transaction"),
                     col('"gateway_region_cd"').alias("gateway_region"),
                     col("_SOURCE_FILE").alias("_bronze_source_file"),
                     col("_LOAD_TIMESTAMP").alias("_bronze_load_timestamp"),
                     current_timestamp().alias("_silver_load_timestamp"),
                     lit("payments").alias("source_system")
                 ))

    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.PAY_TRANSACTIONS")

    rows_inserted = silver_df.count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.PAY_TRANSACTIONS")

    EXECUTION_STATS.append({
        'table': 'PAY_TRANSACTIONS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': duplicates_removed
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PAY_TRANSACTIONS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.PAY_TRANSACTIONS")
print("─" * 80)

validate_row_counts(session, "PAY_TRANSACTIONS", expected_min=0)

# Check payment status distribution
status_dist = (session.table(f"{SILVER_SCHEMA}.PAY_TRANSACTIONS")
               .group_by("payment_status")
               .agg(spark_count("*").alias("count"))
               .sort(col("count").desc()))

print("\n  Payment status distribution:")
status_dist.show()

print("─" * 80)

---

## Shipments

**Source**: `BRONZE.SHP__SHIPMENTS` (XML)

**Transformations**:
- Extract shipment data from XML
- Convert weight units to kg (handle GRM/KGM mix)
- Convert Y/N flags to BOOLEAN
- Handle missing ActualDeliveryDateTimeLocal (absent tag for in-transit)

**Target**: `SILVER.SHP_SHIPMENTS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHP_SHIPMENTS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.SHP__SHIPMENTS → SILVER.SHP_SHIPMENTS")
print("─" * 80)

watermark = get_incremental_watermark(session, "SHP_SHIPMENTS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.SHP__SHIPMENTS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.SHP_SHIPMENTS (
        shipment_id             VARCHAR(50),
        order_id                INTEGER,
        order_source_system     VARCHAR(20),
        origin_facility         VARCHAR(20),
        pickup_datetime         TIMESTAMP_NTZ,
        delivery_datetime       TIMESTAMP_NTZ,
        package_weight_kg       DECIMAL(10, 3),
        weight_unit_original    VARCHAR(10),
        shipment_status         VARCHAR(50),
        signature_required      BOOLEAN,
        insurance_value         DECIMAL(10, 2),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()

    silver_df = session.sql(f"""
    SELECT
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<ShipmentReferenceIdentifier>[^<]*</ShipmentReferenceIdentifier>'), '<[^>]*>', '')::VARCHAR AS shipment_id,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<OrderReferenceIdentifier>[^<]*</OrderReferenceIdentifier>'), '<[^>]*>', '')::INTEGER AS order_id,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<OrderSourceSystem>[^<]*</OrderSourceSystem>'), '<[^>]*>', '')::VARCHAR AS order_source_system,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<OriginFacilityCode>[^<]*</OriginFacilityCode>'), '<[^>]*>', '')::VARCHAR AS origin_facility,
        TRY_TO_TIMESTAMP(
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<ShipmentPickupDateTimeLocal>[^<]*</ShipmentPickupDateTimeLocal>'), '<[^>]*>', '')
        ) AS pickup_datetime,
        TRY_TO_TIMESTAMP(
            REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<ActualDeliveryDateTimeLocal>[^<]*</ActualDeliveryDateTimeLocal>'), '<[^>]*>', '')
        ) AS delivery_datetime,
        CASE
            WHEN REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WeightUnitCode>[^<]*</WeightUnitCode>'), '<[^>]*>', '') = 'GRM'
            THEN REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WeightValue>[^<]*</WeightValue>'), '<[^>]*>', '')::DECIMAL(10,3) / 1000
            ELSE REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WeightValue>[^<]*</WeightValue>'), '<[^>]*>', '')::DECIMAL(10,3)
        END AS package_weight_kg,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<WeightUnitCode>[^<]*</WeightUnitCode>'), '<[^>]*>', '')::VARCHAR AS weight_unit_original,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<CurrentStatusCode>[^<]*</CurrentStatusCode>'), '<[^>]*>', '')::VARCHAR AS shipment_status,
        CASE
            WHEN REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<IsSignatureRequiredFlag>[^<]*</IsSignatureRequiredFlag>'), '<[^>]*>', '') = 'Y'
            THEN TRUE ELSE FALSE
        END AS signature_required,
        REGEXP_REPLACE(REGEXP_SUBSTR(chunk.VALUE, '<InsuranceValueAmount>[^<]*</InsuranceValueAmount>'), '<[^>]*>', '')::DECIMAL(10,2) AS insurance_value,
        bronze._SOURCE_FILE AS _bronze_source_file,
        bronze._LOAD_TIMESTAMP AS _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'shipping' AS source_system
    FROM {BRONZE_SCHEMA}.SHP__SHIPMENTS bronze,
         TABLE(SPLIT_TO_TABLE(bronze.RAW_DATA::VARCHAR, '</ShipmentRecord>')) chunk
    WHERE chunk.VALUE LIKE '%<ShipmentReferenceIdentifier>%'
      AND bronze._LOAD_TIMESTAMP > '{watermark}'
    """)

    silver_df.write.mode("append").save_as_table(f"{SILVER_SCHEMA}.SHP_SHIPMENTS")

    rows_inserted = session.table(f"{SILVER_SCHEMA}.SHP_SHIPMENTS").count()
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.SHP_SHIPMENTS")

    EXECUTION_STATS.append({
        'table': 'SHP_SHIPMENTS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SHP_SHIPMENTS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.SHP_SHIPMENTS")
print("─" * 80)

validate_row_counts(session, "SHP_SHIPMENTS", expected_min=0)

# Check weight conversion
grm_count = (session.table(f"{SILVER_SCHEMA}.SHP_SHIPMENTS")
             .filter(col("weight_unit_original") == lit("GRM"))
             .count())
kgm_count = (session.table(f"{SILVER_SCHEMA}.SHP_SHIPMENTS")
             .filter(col("weight_unit_original") == lit("KGM"))
             .count())

print(f"  ℹ Weight units: {grm_count:,} GRM, {kgm_count:,} KGM (all normalized to kg)")

show_sample_data(session, "SHP_SHIPMENTS", 3)
print("─" * 80)

---

## Support Tickets

**Source**: `BRONZE.SUP__SUPPORT_TICKETS` (CSV with special char column names)

**Transformations**:
- Handle column names with spaces and special chars
- Parse non-standard datetime format
- Handle empty string for open tickets (Solved at)
- Calculate resolution hours
- Convert tags string to cleaner format

**Target**: `SILVER.SUP_TICKETS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUP_TICKETS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.SUP__SUPPORT_TICKETS → SILVER.SUP_TICKETS")
print("─" * 80)

watermark = get_incremental_watermark(session, "SUP_TICKETS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.SUP__SUPPORT_TICKETS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.SUP_TICKETS (
        ticket_id               INTEGER,
        requester_name          VARCHAR(200),
        requester_email         VARCHAR(200),
        assignee                VARCHAR(100),
        tags                    VARCHAR(500),
        order_source            VARCHAR(20),
        ticket_status           VARCHAR(20),
        order_id                INTEGER,
        created_datetime        TIMESTAMP_NTZ,
        solved_datetime         TIMESTAMP_NTZ,
        first_reply_hours       DECIMAL(10, 2),
        csat_score              VARCHAR(20),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform - need to use SQL for complex date parsing
    transform_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.SUP_TICKETS
    SELECT
        "Ticket ID"::INTEGER AS ticket_id,
        "Requester"::VARCHAR AS requester_name,
        LOWER(TRIM("Requester Email"::VARCHAR)) AS requester_email,
        "Assignee"::VARCHAR AS assignee,
        "Tags"::VARCHAR AS tags,
        "Order Source"::VARCHAR AS order_source,
        "Status"::VARCHAR AS ticket_status,
        "Order Reference"::INTEGER AS order_id,
        TO_TIMESTAMP("Created at"::VARCHAR, 'MMMM DD YYYY HH12:MI AM TZD') AS created_datetime,
        CASE 
            WHEN NULLIF("Solved at", '') IS NULL THEN NULL
            ELSE TO_TIMESTAMP("Solved at"::VARCHAR, 'MMMM DD YYYY HH12:MI AM TZD')
        END AS solved_datetime,
        "First Reply (hrs)"::DECIMAL(10, 2) AS first_reply_hours,
        "CSAT Score"::VARCHAR AS csat_score,
        _SOURCE_FILE AS _bronze_source_file,
        _LOAD_TIMESTAMP AS _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'support' AS source_system
    FROM {BRONZE_SCHEMA}.SUP__SUPPORT_TICKETS
    WHERE _LOAD_TIMESTAMP > '{watermark}'
    """
    
    session.sql(transform_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.SUP_TICKETS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.SUP_TICKETS")
    
    EXECUTION_STATS.append({
        'table': 'SUP_TICKETS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUP_TICKETS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.SUP_TICKETS")
print("─" * 80)

validate_row_counts(session, "SUP_TICKETS", expected_min=0)

# Check ticket status distribution
status_dist = (session.table(f"{SILVER_SCHEMA}.SUP_TICKETS")
               .group_by("ticket_status")
               .agg(spark_count("*").alias("count")))

print("\n  Ticket status distribution:")
status_dist.show()

print("─" * 80)

---

## Marketing Campaigns

**Source**: `BRONZE.MKTG__CAMPAIGNS` (CSV with special char column names)

**Transformations**:
- Handle column names with $, #, /, %, ()
- Parse mixed date formats in same column
- Handle NULL CPC/CPA when clicks/conversions are zero

**Target**: `SILVER.MKTG_CAMPAIGNS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MKTG_CAMPAIGNS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.MKTG__CAMPAIGNS → SILVER.MKTG_CAMPAIGNS")
print("─" * 80)

watermark = get_incremental_watermark(session, "MKTG_CAMPAIGNS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.MKTG__CAMPAIGNS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.MKTG_CAMPAIGNS (
        campaign_id             VARCHAR(50),
        campaign_name           VARCHAR(200),
        channel                 VARCHAR(50),
        target_segment          VARCHAR(100),
        date                    DATE,
        start_date              DATE,
        end_date                DATE,
        amount_spent            DECIMAL(10, 2),
        impressions             INTEGER,
        clicks                  INTEGER,
        conversions             INTEGER,
        ctr_percent             DECIMAL(10, 4),
        cpc_amount              DECIMAL(10, 4),
        cpa_amount              DECIMAL(10, 4),
        campaign_status         VARCHAR(20),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform - use SQL for mixed date format handling
    transform_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.MKTG_CAMPAIGNS
    SELECT
        "Campaign ID"::VARCHAR AS campaign_id,
        "Campaign Name"::VARCHAR AS campaign_name,
        "Channel / Platform"::VARCHAR AS channel,
        "Target Segment"::VARCHAR AS target_segment,
        COALESCE(
            TRY_TO_DATE("Date"::VARCHAR, 'YYYY-MM-DD'),
            TRY_TO_DATE("Date"::VARCHAR, 'MM/DD/YYYY')
        ) AS date,
        TRY_TO_DATE("Start Date"::VARCHAR, 'YYYY-MM-DD') AS start_date,
        TRY_TO_DATE("End Date"::VARCHAR, 'YYYY-MM-DD') AS end_date,
        "$ Spent"::DECIMAL(10, 2) AS amount_spent,
        "# Impressions"::INTEGER AS impressions,
        "Clicks (total)"::INTEGER AS clicks,
        "# Conversions"::INTEGER AS conversions,
        "CTR %"::DECIMAL(10, 4) AS ctr_percent,
        "CPC ($)"::DECIMAL(10, 4) AS cpc_amount,
        "CPA ($)"::DECIMAL(10, 4) AS cpa_amount,
        "Status"::VARCHAR AS campaign_status,
        _SOURCE_FILE AS _bronze_source_file,
        _LOAD_TIMESTAMP AS _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'marketing' AS source_system
    FROM {BRONZE_SCHEMA}.MKTG__CAMPAIGNS
    WHERE _LOAD_TIMESTAMP > '{watermark}'
    """
    
    session.sql(transform_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.MKTG_CAMPAIGNS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.MKTG_CAMPAIGNS")
    
    EXECUTION_STATS.append({
        'table': 'MKTG_CAMPAIGNS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MKTG_CAMPAIGNS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.MKTG_CAMPAIGNS")
print("─" * 80)

validate_row_counts(session, "MKTG_CAMPAIGNS", expected_min=0)

# Check date parsing worked
null_dates = (session.table(f"{SILVER_SCHEMA}.MKTG_CAMPAIGNS")
              .filter(col("date").is_null())
              .count())
if null_dates == 0:
    print(f"  ✓ All dates parsed successfully")
else:
    print(f"  ⚠ {null_dates} rows have NULL dates (parsing failed)")

show_sample_data(session, "MKTG_CAMPAIGNS", 3)
print("─" * 80)

---

## Product Reviews

**Source**: `BRONZE.REV__REVIEWS` (JSON with schema split)

**Transformations**:
- Handle schema split (review_id < 1000 uses 'stars', >= 1000 uses 'rating')
- Normalize inconsistent SKU formats with regex
- Handle missing verified_purchase field (old records)
- Map category via BRIDGE_CATEGORY

**Target**: `SILVER.REV_REVIEWS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# REV_REVIEWS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.REV__REVIEWS → SILVER.REV_REVIEWS")
print("─" * 80)

watermark = get_incremental_watermark(session, "REV_REVIEWS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.REV__REVIEWS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.REV_REVIEWS (
        review_id               INTEGER,
        product_sku             VARCHAR(50),
        category_code           VARCHAR(10),
        reviewer_handle         VARCHAR(100),
        review_date             DATE,
        star_rating             INTEGER,
        verified_purchase       BOOLEAN,
        verified_source         VARCHAR(20),
        moderation_status       VARCHAR(20),
        helpful_votes           INTEGER,
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform - use SQL for complex SKU normalization and schema split handling
    transform_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.REV_REVIEWS
    SELECT
        RAW_DATA:review_id::INTEGER AS review_id,
        -- Normalize SKU: extract numbers and reformat to SKU-XXXXX
        'SKU-' || LPAD(REGEXP_REPLACE(UPPER(RAW_DATA:product_sku::VARCHAR), '[^0-9]', ''), 5, '0') AS product_sku,
        cat.category_code AS category_code,
        RAW_DATA:reviewer_handle::VARCHAR AS reviewer_handle,
        TRY_TO_DATE(RAW_DATA:review_date::VARCHAR, 'YYYY-MM-DD') AS review_date,
        -- Handle schema split: old records use 'stars', new use 'rating'
        COALESCE(RAW_DATA:rating::INTEGER, RAW_DATA:stars::INTEGER) AS star_rating,
        -- Default to FALSE if field is missing
        COALESCE(RAW_DATA:verified_purchase::BOOLEAN, FALSE) AS verified_purchase,
        RAW_DATA:verified_source::VARCHAR AS verified_source,
        RAW_DATA:platform_metadata.moderation_status::VARCHAR AS moderation_status,
        RAW_DATA:platform_metadata.helpful_votes::INTEGER AS helpful_votes,
        _SOURCE_FILE AS _bronze_source_file,
        _LOAD_TIMESTAMP AS _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'reviews' AS source_system
    FROM {BRONZE_SCHEMA}.REV__REVIEWS
    LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CATEGORY cat
        ON RAW_DATA:category::VARCHAR = cat.source_category
        AND cat.source_system = 'reviews'
    WHERE _LOAD_TIMESTAMP > '{watermark}'
    """
    
    session.sql(transform_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.REV_REVIEWS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.REV_REVIEWS")
    
    EXECUTION_STATS.append({
        'table': 'REV_REVIEWS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# REV_REVIEWS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.REV_REVIEWS")
print("─" * 80)

validate_row_counts(session, "REV_REVIEWS", expected_min=0)

# Check SKU normalization worked
sku_format_check = (session.table(f"{SILVER_SCHEMA}.REV_REVIEWS")
                    .filter(~col("product_sku").like("SKU-%"))
                    .count())
if sku_format_check == 0:
    print(f"  ✓ All SKUs normalized to SKU-XXXXX format")
else:
    print(f"  ✗ {sku_format_check} SKUs not in expected format")

# Check star rating range
invalid_ratings = (session.table(f"{SILVER_SCHEMA}.REV_REVIEWS")
                   .filter((col("star_rating") < 1) | (col("star_rating") > 5))
                   .count())
if invalid_ratings == 0:
    print(f"  ✓ All star ratings in valid range (1-5)")
else:
    print(f"  ✗ {invalid_ratings} reviews with invalid ratings")

show_sample_data(session, "REV_REVIEWS", 3)
print("─" * 80)

---

## App Events

**Source**: `BRONZE.APP__APP_EVENTS` (JSON clickstream data)

**Transformations**:
- Extract event data from JSON
- Convert Unix milliseconds to TIMESTAMP
- Handle guest sessions (NULL userIdentifier)
- Extract product fields (only present for product-related events)

**Target**: `SILVER.APP_EVENTS`

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# APP_EVENTS — Transform & Load
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Transforming: BRONZE.APP__APP_EVENTS → SILVER.APP_EVENTS")
print("─" * 80)

watermark = get_incremental_watermark(session, "APP_EVENTS")
print(f"Watermark: {watermark}")

bronze_df = (session.table(f"{BRONZE_SCHEMA}.APP__APP_EVENTS")
             .filter(col("_LOAD_TIMESTAMP") > lit(watermark)))

new_count = bronze_df.count()
print(f"New Bronze records: {new_count:,}")

if new_count > 0:
    # Create table
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.APP_EVENTS (
        event_id                VARCHAR(50),
        event_type              VARCHAR(50),
        event_timestamp         TIMESTAMP_NTZ,
        is_authenticated        BOOLEAN,
        user_id                 VARCHAR(50),
        guest_session_id        VARCHAR(100),
        session_id              VARCHAR(100),
        product_sku             VARCHAR(50),
        category_code           VARCHAR(10),
        is_vpn_detected         BOOLEAN,
        app_version             VARCHAR(20),
        _bronze_source_file     VARCHAR(500),
        _bronze_load_timestamp  TIMESTAMP_NTZ,
        _silver_load_timestamp  TIMESTAMP_NTZ,
        source_system           VARCHAR(20)
    )
    """).collect()
    
    # Transform - use SQL for category mapping
    transform_sql = f"""
    INSERT INTO {SILVER_SCHEMA}.APP_EVENTS
    SELECT
        RAW_DATA:eventIdentifierUUID::VARCHAR AS event_id,
        RAW_DATA:eventTypeString::VARCHAR AS event_type,
        TO_TIMESTAMP(RAW_DATA:eventTimestampUnixMilliseconds::BIGINT / 1000) AS event_timestamp,
        RAW_DATA:isUserAuthenticatedAtEventTime::BOOLEAN AS is_authenticated,
        RAW_DATA:userIdentifier::VARCHAR AS user_id,
        RAW_DATA:guestSessionIdentifierString::VARCHAR AS guest_session_id,
        RAW_DATA:sessionIdentifierString::VARCHAR AS session_id,
        RAW_DATA:eventProperties.productListingIdentifierString::VARCHAR AS product_sku,
        cat.category_code AS category_code,
        RAW_DATA:geolocationData.isVpnDetected::BOOLEAN AS is_vpn_detected,
        RAW_DATA:applicationVersionString::VARCHAR AS app_version,
        _SOURCE_FILE AS _bronze_source_file,
        _LOAD_TIMESTAMP AS _bronze_load_timestamp,
        CURRENT_TIMESTAMP() AS _silver_load_timestamp,
        'app_events' AS source_system
    FROM {BRONZE_SCHEMA}.APP__APP_EVENTS
    LEFT JOIN {SILVER_SCHEMA}.BRIDGE_CATEGORY cat
        ON RAW_DATA:eventProperties.productCategoryLabel::VARCHAR = cat.source_category
        AND cat.source_system = 'mobile'
    WHERE _LOAD_TIMESTAMP > '{watermark}'
    """
    
    session.sql(transform_sql).collect()
    
    rows_inserted = (session.table(f"{SILVER_SCHEMA}.APP_EVENTS")
                     .filter(col("_bronze_load_timestamp") > lit(watermark))
                     .count())
    
    # Check guest sessions
    guest_count = (session.table(f"{SILVER_SCHEMA}.APP_EVENTS")
                   .filter(col("_bronze_load_timestamp") > lit(watermark))
                   .filter(col("user_id").is_null())
                   .count())
    
    if guest_count > 0:
        pct = (guest_count / rows_inserted) * 100
        print(f"Guest sessions: {guest_count:,} ({pct:.1f}%)")
    
    print(f"✓ Inserted {rows_inserted:,} rows into SILVER.APP_EVENTS")
    
    EXECUTION_STATS.append({
        'table': 'APP_EVENTS',
        'bronze_rows': new_count,
        'silver_rows': rows_inserted,
        'duplicates': 0
    })
else:
    print("No new records to process")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# APP_EVENTS — Validation
# ══════════════════════════════════════════════════════════════════════════════

print("\n✓ Validating SILVER.APP_EVENTS")
print("─" * 80)

validate_row_counts(session, "APP_EVENTS", expected_min=0)

# Event type distribution
event_dist = (session.table(f"{SILVER_SCHEMA}.APP_EVENTS")
              .group_by("event_type")
              .agg(spark_count("*").alias("count"))
              .sort(col("count").desc())
              .limit(10))

print("\n  Top event types:")
event_dist.show()

print("─" * 80)

---

# Bridge Table — Customer Identity

Rebuild the customer identity bridge from all customer tables to enable cross-source customer analysis.

**Purpose**: Map source-specific customer IDs to canonical email addresses

**Sources**:
- SILVER.WEB_CUSTOMERS (customer_id, email_address)
- SILVER.MOB_CUSTOMERS (customer_id, email_address)
- SILVER.WHL_CUSTOMERS (customer_id, email_address)

**Note**: Marketplace doesn't contribute (emails are masked)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# BRIDGE_CUSTOMER_IDENTITY — Rebuild
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "─" * 80)
print("Rebuilding: SILVER.BRIDGE_CUSTOMER_IDENTITY")
print("─" * 80)

session.sql(f"TRUNCATE TABLE {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY").collect()

rebuild_sql = f"""
INSERT INTO {SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY
SELECT
    email_address AS canonical_customer_id,
    source_system,
    source_customer_id,
    MIN(_silver_load_timestamp)::DATE AS first_seen_date,
    MAX(_silver_load_timestamp)::DATE AS last_seen_date,
    TRUE AS is_active
FROM (
    SELECT email_address, customer_id::VARCHAR AS source_customer_id, source_system, _silver_load_timestamp 
    FROM {SILVER_SCHEMA}.WEB_CUSTOMERS

    UNION ALL

    SELECT email_address, customer_id::VARCHAR AS source_customer_id, source_system, _silver_load_timestamp 
    FROM {SILVER_SCHEMA}.MOB_CUSTOMERS
    WHERE customer_id IS NOT NULL

    UNION ALL

    SELECT email_address, customer_id::VARCHAR AS source_customer_id, source_system, _silver_load_timestamp 
    FROM {SILVER_SCHEMA}.WHL_CUSTOMERS
)
GROUP BY email_address, source_system, source_customer_id
"""

session.sql(rebuild_sql).collect()

bridge_count = session.table(f"{SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY").count()
print(f"✓ Rebuilt BRIDGE_CUSTOMER_IDENTITY with {bridge_count:,} identity mappings")

print("\nSample identity mappings:")
session.table(f"{SILVER_SCHEMA}.BRIDGE_CUSTOMER_IDENTITY").limit(5).show()

print("─" * 80)

---

# Execution Summary

Final statistics and verification of the incremental load run.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Execution Summary
# ══════════════════════════════════════════════════════════════════════════════

RUN_END = datetime.now(timezone.utc)
TOTAL_DURATION = (RUN_END - RUN_START).total_seconds()

print("\n" + "=" * 80)
print("SILVER LAYER INCREMENTAL LOAD — EXECUTION SUMMARY")
print("=" * 80)
print(f"Run started:  {RUN_START.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Run ended:    {RUN_END.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Duration:     {TOTAL_DURATION:.1f} seconds")
print("=" * 80)

# Display table statistics
if EXECUTION_STATS:
    print(f"\nTable Transformation Statistics:\n")
    print(f"{'TABLE':<25} {'BRONZE':<12} {'SILVER':<12} {'DUPLICATES':<12}")
    print("─" * 65)
    
    total_bronze = 0
    total_silver = 0
    total_dupes = 0
    
    for stat in EXECUTION_STATS:
        print(f"{stat['table']:<25} {stat['bronze_rows']:<12,} {stat['silver_rows']:<12,} {stat['duplicates']:<12,}")
        total_bronze += stat['bronze_rows']
        total_silver += stat['silver_rows']
        total_dupes += stat['duplicates']
    
    print("─" * 65)
    print(f"{'TOTAL':<25} {total_bronze:<12,} {total_silver:<12,} {total_dupes:<12,}")
    print("=" * 80)
else:
    print("\nNo new data processed in this run")
    print("=" * 80)

# Verify all Silver tables exist
print("\nSilver Table Verification:\n")
tables = session.sql(f"SHOW TABLES IN SCHEMA {SILVER_SCHEMA}").collect()
silver_data_tables = [
    'WEB_ORDERS', 'WEB_ORDER_ITEMS', 'WEB_CUSTOMERS',
    'MOB_ORDERS', 'MOB_ORDER_ITEMS', 'MOB_CUSTOMERS',
    'WHL_ORDERS', 'WHL_ORDER_ITEMS', 'WHL_CUSTOMERS',
    'MKT_ORDERS', 'PAY_TRANSACTIONS', 'SHP_SHIPMENTS',
    'SUP_TICKETS', 'MKTG_CAMPAIGNS', 'REV_REVIEWS', 'APP_EVENTS'
]

print(f"{'TABLE NAME':<30} {'ROW COUNT':>15}")
print("─" * 47)

for table_name in silver_data_tables:
    try:
        count = session.table(f"{SILVER_SCHEMA}.{table_name}").count()
        print(f"✓ {table_name:<28} {count:>15,}")
    except Exception as e:
        print(f"✗ {table_name:<28} {'ERROR':>15}")

print("─" * 47)
print("\n✓ Silver layer incremental load complete!")
print("=" * 80)

In [ ]:
import t